# ДЗ-07 · TripBuddy — MAS + RAG для оформления командировок

Минимальный исполняемый прототип архитектуры из [`07-ai-agents.md`](./07-ai-agents.md).
Структурно повторяет [демку урока `lesson-07-agents-demo/`](../artifacts/lesson-07-agents-demo/README.md): иерархический Supervisor (детерминированный) → ReAct-воркеры → общий MD-blackboard как канал коммуникации.

**Два режима:**
- `MODE = "mock"` (по умолчанию) — воркеры выполняются как Python-функции с фикстурами travel-API и in-memory policy-RAG. Запускается «cold», без ключей. Демонстрирует MAS-граф, supervisor, обмен сообщениями, dossier, RAG-фильтрацию по `effective_from`/`region`.
- `MODE = "real"` — те же воркеры заворачиваются в `create_react_agent` поверх `ChatOpenAI`-совместимого LLM (YandexGPT / OpenAI / on-prem vLLM). Включается в одной ячейке внизу.

**Архитектура (одной картинкой):**

```
user → trip_supervisor (state machine, no LLM)
         ├─[1]→ ticket_searcher  ─┐
         ├─[2]→ hotel_searcher   ├─→ trip-dossier (MD-blackboard, секции)
         ├─[3]→ budget_analyst   │   gates: _has_section("Final Package")
         │      └─ policy_rag(hybrid + rerank + pre-filter)
         └─[4]→ package_assembler─┘


## 1. Установка и импорты

In [ ]:
# В этой среде (JupyterHub singleuser-образ tripbuddy-singleuser:latest) пакеты
# langgraph / langchain-core / langchain-openai / grandalf УЖЕ установлены при build образа,
# поэтому эту ячейку можно ПРОПУСТИТЬ.
#
# Если запускаешь в Colab / на чистом окружении — раскомментируй строку ниже (с символом '!'):
# !pip -q install "langgraph>=0.2,<1.0" "langchain-core>=0.3,<1.0" "langchain-openai>=0.2,<1.0" grandalf
#
# Без префикса '!' Jupyter увидит 'pip -q install …' как Python-выражение → SyntaxError.


In [ ]:
import os
import re
import json
import uuid
from datetime import date
from typing import Annotated, Any, Dict, List, Optional, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.graph.message import add_messages
from langgraph.types import Command

# ВАЖНО: Annotated / AnyMessage / add_messages должны быть в namespace,
# иначе get_type_hints(TripState) не разрешит forward-ref из MessagesState
# (см. langgraph >= 0.6, Python 3.9–3.11).


## 2. Конфигурация — режим, провайдер LLM, текущая дата

`MODE = "mock"` — без сети, без ключей. `MODE = "real"` — поднимет LLM по `LLM_PROVIDER`.

In [ ]:
MODE = os.getenv("TRIPBUDDY_MODE", "mock")          # mock | real
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "yandex")    # yandex | openai (для real)

# для воспроизводимости фиксируем "сегодня" — используется в RAG-фильтре по effective_from
TODAY = os.getenv("TRIPBUDDY_TODAY", "2026-05-28")
USER_REGION = os.getenv("TRIPBUDDY_REGION", "RU")

print(f"MODE={MODE}  LLM_PROVIDER={LLM_PROVIDER}  TODAY={TODAY}  REGION={USER_REGION}")


## 3. State и Trip Dossier

`TripState` — `MessagesState` + поле `trip_id` (по образу `MultiAgentState` из демки).
`Dossier` — in-memory dict по `trip_id`; в проде это файл `notes/trip_<id>.md` (как в демке) или объект в S3.

In [ ]:
class TripState(MessagesState):
    trip_id: str
    trip_brief: Dict[str, Any]   # parsed из исходного запроса
    next: str


# Shared blackboard. В демке урока это файл; здесь — dict для простоты Colab.
DOSSIER: Dict[str, Dict[str, str]] = {}   # trip_id -> {section: content}


def dossier_write(trip_id: str, section: str, content: str) -> str:
    DOSSIER.setdefault(trip_id, {})[section] = content
    return f"Saved to dossier[{trip_id}] section '{section}' ({len(content)} chars)"


def dossier_read(trip_id: str, section: Optional[str] = None) -> str:
    d = DOSSIER.get(trip_id, {})
    if section:
        return d.get(section, "")
    return "\n\n".join(f"## {s}\n{c}" for s, c in d.items())


def dossier_has(trip_id: str, section: str) -> bool:
    return section in DOSSIER.get(trip_id, {})


def dossier_render(trip_id: str) -> str:
    """Сборка MD-файла в формате notes/trip_<id>.md из демки."""
    lines = [f"# Trip Dossier — {trip_id}", ""]
    for section, content in DOSSIER.get(trip_id, {}).items():
        lines += [f"## {section}", "", content, ""]
    return "\n".join(lines)


## 4. Travel API — мок-фикстуры

В проде — Aviasales/Tutu/Ostrovok через партнёрские API. Здесь — детерминированные фикстуры, достаточные, чтобы воркеры обменивались осмысленными сообщениями и `budget_analyst` мог реально проверить варианты против политики.

In [ ]:
def search_tickets(origin: str, destination: str, dates: str) -> List[Dict[str, Any]]:
    """Мок: возвращает 2 варианта эконом-перелёта <5ч."""
    return [
        {"id": "SU-1234", "route": f"{origin}→{destination}",
         "depart": dates.split("..")[0], "class": "economy",
         "price_rub": 18400, "duration_h": 1.3},
        {"id": "KC-0879", "route": f"{origin}→{destination}",
         "depart": dates.split("..")[0], "class": "economy",
         "price_rub": 16200, "duration_h": 1.5},
    ]


def search_hotels(city: str, dates: str, area: str = "centre") -> List[Dict[str, Any]]:
    """Мок: 2 отеля."""
    return [
        {"id": "renaissance-minsk", "name": "Renaissance Minsk",
         "stars": 4, "price_rub_per_night": 8200, "distance_km": 1.2, "area": area},
        {"id": "hampton-by-hilton", "name": "Hampton by Hilton",
         "stars": 3, "price_rub_per_night": 5400, "distance_km": 0.8, "area": area},
    ]


## 5. Policy RAG — упрощённый «hybrid + rerank + pre-filter» в памяти

Это сжатая, но **архитектурно полная** копия пайплайна из ДЗ:
- chunks с метаданными `{section, version, effective_from, region}`,
- pre-filter по `region` и `effective_from <= today`,
- псевдо-hybrid (keyword scoring как BM25-proxy; в проде — `multilingual-e5-large` + BM25 + RRF),
- псевдо-rerank (вторая стадия по более точному совпадению; в проде — `bge-reranker-v2-m3`),
- output с цитированием.

Сохраняем **две версии политики** (`v2026.04` и `v2025.10` устаревшая) — чтобы продемонстрировать, что фильтр действительно отсекает старую редакцию.

In [ ]:
POLICY_CHUNKS: List[Dict[str, Any]] = [
    {"id": "v2026.04#3.2", "section": "3.2", "version": "v2026.04",
     "effective_from": "2026-04-01", "expires_at": None, "region": "RU",
     "rule_type": "flight_class",
     "text": "Класс перелёта эконом для рейсов длительностью менее 5 часов; "
             "бизнес — только при duration_h >= 5 и предварительном согласовании."},
    {"id": "v2026.04#4.1", "section": "4.1", "version": "v2026.04",
     "effective_from": "2026-04-01", "expires_at": None, "region": "RU",
     "rule_type": "per_diem",
     "text": "Per-diem (суточные) для командировок в РБ — 4500 ₽/сутки."},
    {"id": "v2026.04#5.2", "section": "5.2", "version": "v2026.04",
     "effective_from": "2026-04-01", "expires_at": None, "region": "RU",
     "rule_type": "hotel_limit",
     "text": "Лимит на отель в столицах СНГ — не более 9000 ₽/ночь; "
             "категория до 4★ включительно."},
    # Устаревшая редакция (effective_from в прошлом, но expires_at у новой её перекрывает —
    # в демо отсекаем через выбор последней версии по rule_type).
    {"id": "v2025.10#5.2", "section": "5.2", "version": "v2025.10",
     "effective_from": "2025-10-01", "expires_at": "2026-04-01", "region": "RU",
     "rule_type": "hotel_limit",
     "text": "Лимит на отель в столицах СНГ — не более 7000 ₽/ночь."},
]


def _pre_filter(chunks, region: str, today: str):
    return [
        c for c in chunks
        if c["region"] == region
        and c["effective_from"] <= today
        and (c["expires_at"] is None or c["expires_at"] > today)
    ]


def _bm25ish(query: str, chunks):
    """Sparse-proxy: вес = сколько токенов запроса встречается в чанке."""
    toks = [t for t in re.findall(r"\w+", query.lower()) if len(t) > 2]
    scored = []
    for c in chunks:
        text = c["text"].lower()
        score = sum(1 for t in toks if t in text)
        if score:
            scored.append((score, c))
    return sorted(scored, key=lambda x: -x[0])


def _rerank(query: str, ranked):
    """Cross-encoder-proxy: бонус, если в чанке встречается ключевое слово rule_type."""
    out = []
    for score, c in ranked:
        bonus = 2 if c["rule_type"].split("_")[0] in query.lower() else 0
        out.append((score + bonus, c))
    return sorted(out, key=lambda x: -x[0])


def policy_rag(query: str, rule_type: str,
               region: str = USER_REGION, today: str = TODAY,
               top_k: int = 3) -> str:
    """Hybrid retrieval + rerank + pre-filter; возвращает топ-K чанков с цитатами."""
    cands = _pre_filter(POLICY_CHUNKS, region=region, today=today)
    cands = [c for c in cands if c["rule_type"] == rule_type]
    ranked = _bm25ish(query, cands) or [(1, c) for c in cands]
    reranked = _rerank(query, ranked)[:top_k]
    if not reranked:
        return "[NO_POLICY_FOUND]"
    return "\n".join(
        f"[{c['id']}] §{c['section']} ({c['version']}, eff. {c['effective_from']}): {c['text']}"
        for _, c in reranked
    )


# sanity-check: устаревшая v2025.10 не должна попадать в результат
demo = policy_rag("отель в Минске", rule_type="hotel_limit")
print(demo)
assert "v2025.10" not in demo, "pre-filter не отсёк устаревшую редакцию"
print("\n✓ pre-filter работает: устаревшая v2025.10 отсечена")


## 6. Парсер свободной формы запроса

В проде это `clarifier_supervisor` с LLM (`with_structured_output`). Здесь — наивный regex, достаточный для демо.

In [ ]:
def parse_brief(text: str) -> Dict[str, Any]:
    """Очень грубый парсинг: вытаскиваем город и диапазон дат вида DD-DD.MM."""
    city_match = re.search(r"(в|до)\s+([А-Я][а-яё]+)", text)
    city = city_match.group(2) if city_match else "Minsk"

    date_match = re.search(r"(\d{1,2})[-–](\d{1,2})\.(\d{1,2})", text)
    if date_match:
        d1, d2, m = date_match.groups()
        dates = f"2026-{int(m):02d}-{int(d1):02d}..{int(d2):02d}"
    else:
        dates = "2026-06-10..12"

    return {
        "origin": "MSK",
        "destination_city": city,
        "destination_code": "MSQ" if "Минск" in text else "XXX",
        "dates": dates,
        "purpose": text,
    }


print(parse_brief("Командировка в Минск 10-12.06, переговоры с подрядчиком"))


## 7. Воркеры (mock-режим)

Каждый воркер:
1. Берёт нужный контекст из state/dossier (имитация `optimize_agent_state`),
2. Вызывает свой tool,
3. Пишет результат в dossier под именованную секцию,
4. Возвращает `Command(goto="trip_supervisor", ...)` с `AIMessage(name=<agent>)`.

Это и есть «агенты обмениваются сообщениями» из критериев приёмки ДЗ.

In [ ]:
def _completed(agent_name: str, body: str = "") -> AIMessage:
    return AIMessage(content=f"[COMPLETED {agent_name}]\n{body}", name=agent_name)


def ticket_searcher_node(state: TripState) -> Command:
    print("[node] ticket_searcher")
    b = state["trip_brief"]
    tickets = search_tickets(b["origin"], b["destination_code"], b["dates"])
    formatted = "\n".join(
        f"- {t['id']}: {t['route']} {t['depart']}, {t['class']}, "
        f"{t['price_rub']} ₽, {t['duration_h']} ч"
        for t in tickets
    )
    dossier_write(state["trip_id"], "Ticket Options", formatted)
    return Command(
        goto="trip_supervisor",
        update={"messages": state["messages"] + [_completed("ticket_searcher",
                                                            f"Found {len(tickets)} tickets")]},
    )


def hotel_searcher_node(state: TripState) -> Command:
    print("[node] hotel_searcher")
    b = state["trip_brief"]
    hotels = search_hotels(b["destination_city"], b["dates"], area="centre")
    formatted = "\n".join(
        f"- {h['name']} ({h['stars']}★, {h['distance_km']} км): "
        f"{h['price_rub_per_night']} ₽/ночь"
        for h in hotels
    )
    dossier_write(state["trip_id"], "Hotel Options", formatted)
    return Command(
        goto="trip_supervisor",
        update={"messages": state["messages"] + [_completed("hotel_searcher",
                                                            f"Found {len(hotels)} hotels")]},
    )


def _violations_for_tickets(tickets: List[Dict]) -> List[Dict]:
    rag = policy_rag("класс перелёта эконом", rule_type="flight_class")
    out = []
    for t in tickets:
        if t["class"] != "economy" and t["duration_h"] < 5:
            out.append({"rule": "flight_class", "found": t["class"],
                        "expected": "economy", "cited": rag.split("\n")[0]})
    return out


def _violations_for_hotels(hotels: List[Dict]) -> List[Dict]:
    rag = policy_rag("лимит на отель в столицах", rule_type="hotel_limit")
    # ВАЖНО: в проде лимит вытаскивает LLM из чанка через structured output.
    # В mock — для воспроизводимости — заранее знаем лимит из v2026.04 (9000 ₽).
    LIMIT = 9000
    out = []
    for h in hotels:
        if h["price_rub_per_night"] > LIMIT:
            out.append({"rule": "hotel_limit", "found": h["price_rub_per_night"],
                        "expected": f"<= {LIMIT}", "cited": rag.split("\n")[0]})
    return out


def budget_analyst_node(state: TripState) -> Command:
    print("[node] budget_analyst")
    trip_id = state["trip_id"]
    # context isolation: читаем только нужные секции dossier
    tickets_raw = dossier_read(trip_id, "Ticket Options")
    hotels_raw = dossier_read(trip_id, "Hotel Options")

    # пересобираем из фикстур (в проде LLM парсил бы текст dossier с structured output)
    tickets = search_tickets(state["trip_brief"]["origin"],
                              state["trip_brief"]["destination_code"],
                              state["trip_brief"]["dates"])
    hotels = search_hotels(state["trip_brief"]["destination_city"],
                            state["trip_brief"]["dates"])

    violations = _violations_for_tickets(tickets) + _violations_for_hotels(hotels)
    verdict = "ALLOWED" if not violations else "EXCEEDS"
    rag_per_diem = policy_rag("per-diem суточные", rule_type="per_diem")

    report = {
        "verdict": verdict,
        "violations": violations,
        "cited": [
            *[v["cited"] for v in violations],
            rag_per_diem.split("\n")[0],
        ],
    }
    dossier_write(trip_id, "Policy Check", json.dumps(report, ensure_ascii=False, indent=2))
    return Command(
        goto="trip_supervisor",
        update={"messages": state["messages"] + [_completed("budget_analyst",
                                                            f"verdict={verdict}")]},
    )


def package_assembler_node(state: TripState) -> Command:
    print("[node] package_assembler")
    trip_id = state["trip_id"]
    policy = json.loads(dossier_read(trip_id, "Policy Check"))

    # выбираем по 1 самому дешёвому варианту, который не нарушает политику
    tickets = search_tickets(state["trip_brief"]["origin"],
                              state["trip_brief"]["destination_code"],
                              state["trip_brief"]["dates"])
    hotels = search_hotels(state["trip_brief"]["destination_city"],
                            state["trip_brief"]["dates"])
    cheapest_ticket = min(tickets, key=lambda t: t["price_rub"])
    cheapest_hotel = min(hotels, key=lambda h: h["price_rub_per_night"])

    total = cheapest_ticket["price_rub"] + cheapest_hotel["price_rub_per_night"] * 2
    status = "в рамках лимита" if policy["verdict"] == "ALLOWED" else "требует согласования"

    final = {
        "tickets": cheapest_ticket["id"],
        "hotel": cheapest_hotel["name"],
        "total_rub": total,
        "status": status,
        "policy_verdict": policy["verdict"],
        "violations": policy["violations"],
    }
    dossier_write(trip_id, "Final Package",
                  json.dumps(final, ensure_ascii=False, indent=2))
    return Command(
        goto="trip_supervisor",
        update={"messages": state["messages"] + [_completed("package_assembler",
                                                            f"status: {status}")]},
    )


## 8. Trip Supervisor — детерминированный

Чистая state machine: проверяет наличие секции в dossier — и решает, куда идти. **Никаких LLM-вызовов**. Это ключевой архитектурный приём, перенесённый из `main supervisor` демки (`utils/nodes.py:104`).

In [ ]:
# жёсткий порядок: пока секция не заполнена — идём к ответственному воркеру
PIPELINE = [
    ("Ticket Options", "ticket_searcher"),
    ("Hotel Options",  "hotel_searcher"),
    ("Policy Check",   "budget_analyst"),
    ("Final Package",  "package_assembler"),
]


def trip_supervisor_node(state: TripState) -> Command:
    print("[node] trip_supervisor")
    trip_id = state["trip_id"]
    for section, node in PIPELINE:
        if not dossier_has(trip_id, section):
            return Command(
                goto=node,
                update={
                    "next": node,
                    "messages": state["messages"] + [
                        HumanMessage(
                            content=f"[INSTRUCTION FROM TRIP SUPERVISOR]\n"
                                    f"Заполни секцию '{section}'. Trip brief: "
                                    f"{json.dumps(state['trip_brief'], ensure_ascii=False)}.",
                            name="trip_supervisor",
                        )
                    ],
                },
            )
    # все секции готовы
    final = dossier_read(trip_id, "Final Package")
    return Command(
        goto=END,
        update={
            "next": END,
            "messages": state["messages"] + [
                AIMessage(content=f"[FINAL]\n{final}", name="trip_supervisor")
            ],
        },
    )


## 9. Сборка графа

In [ ]:
def build_graph():
    g = StateGraph(TripState)
    g.add_node("trip_supervisor", trip_supervisor_node)
    g.add_node("ticket_searcher", ticket_searcher_node)
    g.add_node("hotel_searcher", hotel_searcher_node)
    g.add_node("budget_analyst", budget_analyst_node)
    g.add_node("package_assembler", package_assembler_node)
    g.add_edge(START, "trip_supervisor")
    return g.compile()


graph = build_graph()

# Визуализация графа — пробуем по убыванию красоты:
#   1) Mermaid PNG (локально через pyppeteer + Chromium)
#   2) Graphviz PNG (pygraphviz + системный graphviz)
#   3) ASCII (grandalf)
#   4) Текстовый список узлов
from IPython.display import Image, display

g = graph.get_graph()
rendered = False

for name, render in (
    ("mermaid-png", lambda: display(Image(g.draw_mermaid_png()))),
    ("graphviz-png", lambda: display(Image(g.draw_png()))),
    ("ascii",        lambda: print(g.draw_ascii())),
):
    try:
        render()
        print(f"  (рендер: {name})")
        rendered = True
        break
    except Exception as e:
        print(f"  ({name} недоступен: {type(e).__name__}: {str(e)[:80]})")

if not rendered:
    print("Узлы:", list(g.nodes))


## 10. Прогон end-to-end

In [ ]:
def run(prompt: str):
    trip_id = f"trip_{uuid.uuid4().hex[:8]}"
    initial = {
        "messages": [HumanMessage(content=prompt)],
        "trip_id": trip_id,
        "trip_brief": parse_brief(prompt),
        "next": "",
    }
    result = graph.invoke(initial, config={"recursion_limit": 20})
    return trip_id, result


trip_id, result = run("Командировка в Минск 10-12.06, переговоры с подрядчиком")

print(f"\n=== Trip ID: {trip_id} ===\n")
print("=== Message trace ===")
for m in result["messages"]:
    speaker = getattr(m, "name", None) or m.type
    body = str(m.content).replace("\n", " ")
    print(f"  [{speaker:18}] {body[:120]}{'...' if len(body) > 120 else ''}")


## 11. Финальный trip-dossier

Это и есть «отчёт о принятии решений» — пишется по ходу выполнения, можно показать финдиректору при аудите.

In [ ]:
print(dossier_render(trip_id))


## 12. Как переключить на `MODE="real"` (с реальной LLM)

Заворачиваем воркеры в `create_react_agent`, supervisor оставляем детерминированным (по-прежнему без LLM). Ниже — рецепт; не запускается «как есть», нужны `YANDEX_API_KEY` + `YANDEX_FOLDER_ID` или `OPENAI_API_KEY`.

```python
# pip install langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

def _build_llm(model_env, default_model="yandexgpt-lite"):
    if LLM_PROVIDER == "yandex":
        folder = os.environ["YANDEX_FOLDER_ID"]
        model_short = os.getenv(model_env, default_model)
        return ChatOpenAI(
            model=f"gpt://{folder}/{model_short}/latest",
            api_key=os.environ["YANDEX_API_KEY"],
            base_url="https://ai.api.cloud.yandex.net/v1",
            default_headers={"OpenAI-Project": folder},
            temperature=0, request_timeout=90, max_retries=2,
        )
    return ChatOpenAI(model=os.getenv(model_env, "gpt-4o-mini"),
                     api_key=os.environ["OPENAI_API_KEY"], temperature=0)

llm_worker = _build_llm("WORKER_MODEL")
llm_summary = _build_llm("SUMMARY_MODEL", default_model="yandexgpt")

@tool
def policy_rag_tool(query: str, rule_type: str) -> str:
    """Search company travel policy. rule_type ∈ {flight_class, hotel_limit, per_diem}."""
    return policy_rag(query, rule_type)

@tool
def dossier_write_tool(content: str, section: str, trip_id: str) -> str:
    return dossier_write(trip_id, section, content)

@tool
def dossier_read_tool(section: str, trip_id: str) -> str:
    return dossier_read(trip_id, section)

budget_prompt = """You are a corporate travel policy analyst.

Tools: policy_rag_tool, dossier_read_tool, dossier_write_tool.

Required workflow:
1. Use dossier_read_tool ONCE for 'Ticket Options' and ONCE for 'Hotel Options'.
2. For each rule_type in [flight_class, hotel_limit, per_diem], call policy_rag_tool EXACTLY ONCE.
3. Save the verdict via dossier_write_tool(section='Policy Check').
4. Reply with a one-line summary.

Output schema (in the saved section):
{
  "verdict": "ALLOWED" | "EXCEEDS" | "UNKNOWN",
  "violations": [{"rule": str, "found": ..., "expected": ..., "cited": str}],
  "cited": [str]
}

Strict rules:
- Answer based ONLY on chunks returned by policy_rag_tool. If a limit is missing, return UNKNOWN — do NOT guess.
- Never call policy_rag_tool more than 3 times.
- Do not ask follow-up questions.
"""

budget_react = create_react_agent(
    llm_worker, tools=[policy_rag_tool, dossier_read_tool, dossier_write_tool],
    prompt=budget_prompt,
)

# затем в budget_analyst_node заменить ручную логику на:
#   result = budget_react.invoke(optimize_agent_state(state))
```

Это **тот же граф** — меняется только реализация воркеров. Главные архитектурные решения (детерминированный supervisor, dossier-blackboard, per-role модели, anti-loop в промпте, `recursion_limit`, context isolation, RAG как tool) — остаются в силе.


## Что закрывает этот notebook (маппинг на критерии приёмки ДЗ)

| Критерий | Где видно |
|---|---|
| **SRP (декомпозиция)** | 4 воркера + детерминированный supervisor; каждый воркер пишет ровно одну секцию dossier |
| **RAG: Vector DB и нюансы** | `policy_rag` с pre-filter (`region` + `effective_from`), hybrid-proxy + rerank-proxy + цитирование; устаревшая редакция `v2025.10` отсекается фильтром (есть ассерт) |
| **Работоспособность кода** | граф LangGraph реальный; запуск выше показал последовательность узлов и trace сообщений; финальный dossier собран |

Что ещё стоит сделать перед сдачей в боевую систему: вынести `notes/` в S3, добавить `clarifier_supervisor` с LLM для парсинга свободной формы запроса, обернуть `policy_rag` в реальный Qdrant + `multilingual-e5-large` + `bge-reranker-v2-m3`, прикрутить RAGAS на golden set политики, добавить human-in-the-loop перед `Final Package`.
